La empresa necesita realizar un modelo de fraude. Tu papel como DS es dar una solución de modelo y
mitigar los riesgos de fraude basada en datos. Los datos para realizar el modelo son datos_fraud.csv.

Una vez que obtengas los datos, utiliza todos tus conocimientos y todos los pasos que creas
necesarios para poder entregar un modelo funcional para predecir si un cliente debería de ser
aprobado o no. Entre los pasos que esperamos ver están:
- EDA
- Pre-procesamiento de los datos
- Entrenamiento del modelo
- Testing de modelo
- Una explicación de cómo pondrías este modelo en producción y que tendrías que estarle
cuidando con el tiempo

In [1]:
#!pip install sweetviz
#!pip install ipywidgets

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv("Archivos_aux/datos_fraud.csv")

In [5]:
import sweetviz as sv
#report = sv.analyze(df)
#report.show_html("Salidas/EDA-datos_fraud.html")

Se hace el analisis estadistico preeliminar de las variables.
- Las variables no cuentan con un nombre mas que el ID, la ficha de tiempo, el monto de la transaccion, y la clasificacion de si es fraude o no. 

- Se cuenta con 32 variables sin descripcion con varianzas menores a 2 en la mayor parte de los casos y distribuciones leptocurtucas con datos atipicos en las colas. (Se realizara revision)

- La variable 31 presenta una varianza mayor a las demas por lo que habra que normalizar para evitar dominancia en el modelo.

- No se cuenta con datos nulos o faltantes en ninguna columna. 

- Existen IDs  repetidos siendo el mayor con una repeticion de 18 veces

- Solo hay 492 casos de fraude de una base de 284,807 siendo menos del 1% de los casos.

In [6]:
df['fecha_hora'] = pd.to_datetime('2026-01-01') + pd.to_timedelta(df['timestamp'], unit='s')

In [9]:
#!pip install xgboost
#!pip install lightgbm

In [10]:
import lightgbm as lgb
import numpy as np
import pandas as pd

# Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)

# Preprocesamiento y Separación
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier

In [ ]:


df['amount'] = df['amount'].astype(np.float64)
# 1. Transformación Logarítmica: np.log1p equivale a log(x + 1)
df['amount_log'] = np.log1p(df['amount'])

df['es_uno_o_menos'] = (df['amount'] <= 1.0).astype(int)

# 3. Flag de cero absoluto (Validaciones sin costo)
df['es_cero'] = (df['amount'] == 0.0).astype(int)

# Categorización por rangos de monto usando deciles (10 grupos del 0 al 9)
with np.errstate(under="ignore"):
  df['deciles'] = pd.qcut(
      df['amount'], q=10, labels=False, duplicates='drop'
  )


# 1. Extraer la hora del día (0 a 23)
# Dado que tu timestamp eran segundos transcurridos:
df['hour'] = ((df['timestamp'] / 3600) % 24).astype(int)

# 2. Extraer el día de la transacción (Día 1 vs Día 2)
df['day'] = ((df['timestamp'] / 86400)).astype(int) + 1

# 3. Flag de horario nocturno / riesgo (Las transacciones de madrugada suelen tener mayor índice de fraude)
# Ejemplo: entre las 00:00 y las 06:00 AM
df['is_night_transaction'] = df['hour'].apply(lambda x: 1 if 0 <= x <= 5 else 0)

# 4. Transformación cíclica de la hora (Seno / Coseno)
# Le enseña al modelo que las 23:59 y las 00:01 están consecutivas y son el mismo momento del día
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)


# ---------------------------------------------------------
# 1. Separación Estratificada (Train / Test)
# ---------------------------------------------------------
X = df.drop(columns=['is_fraud', "transaction_id","amount","timestamp"])  
y = df['is_fraud']

# Separación 80/20 manteniendo el balance de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

cols_to_scale = [
    'amount',
    'variable_31',
    'variable_01',
    'variable_21',
    'variable_06',
    'variable_27',
    'variable_24',
    'variable_28',
    'variable_12',
    'variable_08',
    'variable_22',
]



# 2. Inicializar el transformador
scaler = RobustScaler()

# 3. Sobreescribir las columnas en el mismo DataFrame
X_test[cols_to_scale] = scaler.fit_transform(X_test[cols_to_scale])




In [15]:
X_test.head(2)

,timestamp,variable_01,variable_02,variable_03,variable_04,variable_05,variable_06,variable_07,variable_08,variable_09,...,variable_28,variable_29,variable_30,variable_31,variable_32,fecha_hora,amount_log,es_uno_o_menos,es_cero,deciles
262988,143206.0,-0.056109,-0.027635,-0.488264,0.483944,0.018215,-0.047606,0.564347,0.158763,-0.159223,...,2.001881,-1.660096,-0.779008,-0.740641,-0.088432,2026-01-02 15:46:46,3.610648,0,0,5
11348,34834.0,-0.034688,-0.033755,-0.542984,0.536037,-0.376974,-0.166231,-0.432113,-0.100241,-0.121949,...,-0.840200,-1.846146,0.175126,-0.457885,-0.108018,2026-01-01 09:40:34,3.413455,0,0,5


In [ ]:

# ---------------------------------------------------------
# 2. Transformaciones (Preprocesamiento sin Data Leakage)
# ---------------------------------------------------------
# Para modelos de árboles, transformar amount con log1p facilita la división de nodos.
# Aplicamos la transformación después del split.
X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

if 'amount' in X_train_proc.columns:
    X_train_proc['amount'] = np.log1p(X_train_proc['amount'])
    X_test_proc['amount'] = np.log1p(X_test_proc['amount'])

# Cálculo del peso de balanceo para desbalance severo
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# ---------------------------------------------------------
# 3. Definición de Modelos
# ---------------------------------------------------------
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
}

# ---------------------------------------------------------
# 4. Entrenamiento y Evaluación
# ---------------------------------------------------------
results = []

for name, model in models.items():
    print(f"\n=================== Entrenando {name} ===================")
    model.fit(X_train_proc, y_train)

    # Predicción de probabilidades
    y_pred_proba = model.predict_proba(X_test_proc)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Métricas clave para datasets desbalanceados
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(
        y_test, y_pred_proba
    )  # PR-AUC es vital en fraude

    results.append({'Modelo': name, 'ROC-AUC': roc_auc, 'PR-AUC': pr_auc})

    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC (Average Precision): {pr_auc:.4f}")
    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred, digits=4))

# Resumen de resultados
df_resumen = pd.DataFrame(results).sort_values(by='PR-AUC', ascending=False)
print("\n=== RESUMEN DE RENDIMIENTO ===")
print(df_resumen.to_string(index=False))